# Gemma 4 31B Tool-Calling Fine-Tune

Based on the [official Unsloth Gemma4_(31B)-Text notebook](https://github.com/unslothai/notebooks/blob/main/nb/Gemma4_(31B)-Text.ipynb).

Dataset: [mtita/gemma4-tool-calling-training](https://huggingface.co/datasets/mtita/gemma4-tool-calling-training) (32,893 examples)

**Run All** to start training. Set your HF token in the Save cell at the bottom.


### Installation


In [ ]:
# === Create a clean AMD/Unsloth kernel (doc-aligned path) ===
# This follows the official AMD + Unsloth guidance: isolated env, ROCm PyTorch, then Unsloth.
import os, re, shutil, subprocess, sys
from pathlib import Path
VENV = os.path.expanduser('~/unsloth_gemma4_env')
PY = os.path.join(VENV, 'bin', 'python')
PIP = [PY, '-m', 'pip']

def run(cmd):
    print('\n$ ' + ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout.strip():
        out = r.stdout.strip().splitlines()
        for line in out[-8:]:
            print(line)
    if r.returncode != 0:
        err = r.stderr.strip() or '(no stderr)'
        raise RuntimeError(err)

def detect_rocm_tag():
    candidates = []
    try:
        r = subprocess.run(['amd-smi', 'version'], capture_output=True, text=True)
        candidates.append(r.stdout)
    except FileNotFoundError:
        pass
    try:
        candidates.append(Path('/opt/rocm/.info/version').read_text())
    except Exception:
        pass
    try:
        r = subprocess.run(['hipconfig', '--version'], capture_output=True, text=True)
        candidates.append(r.stdout)
    except FileNotFoundError:
        pass
    for text in candidates:
        m = re.search(r'(?:ROCm version: |HIP version: )([0-9]+)\.([0-9]+)', text)
        if not m:
            m = re.search(r'([0-9]+)\.([0-9]+)', text)
        if m:
            return f'rocm{m.group(1)}.{m.group(2)}'
    return 'rocm7.0'

rocm_tag = detect_rocm_tag()
print('ROCm wheel tag:', rocm_tag)

if os.path.isdir(VENV):
    shutil.rmtree(VENV)
run([sys.executable, '-m', 'venv', VENV])
run(PIP + ['install', '--upgrade', 'pip', 'setuptools', 'wheel', 'ipykernel'])
run(PIP + ['install', '--upgrade', '--force-reinstall', 'torch', 'torchvision', 'torchaudio', '--index-url', f'https://download.pytorch.org/whl/{rocm_tag}'])
run(PIP + ['install', '--no-deps', 'unsloth', 'unsloth-zoo'])
run(PIP + ['install', '--no-deps', 'git+https://github.com/unslothai/unsloth-zoo.git'])
run(PIP + ['install', 'unsloth[amd] @ git+https://github.com/unslothai/unsloth'])
run(PIP + ['install', '--upgrade', 'git+https://github.com/huggingface/transformers.git'])
run(PIP + ['install', '--upgrade', 'timm', 'torchcodec'])
run([PY, '-m', 'ipykernel', 'install', '--user', '--name', 'unsloth-gemma4-amd', '--display-name', 'Python (unsloth-gemma4-amd)'])

print('\nKernel created: Python (unsloth-gemma4-amd)')
print('Next: Kernel -> Change Kernel -> Python (unsloth-gemma4-amd)')
print('Then restart kernel and run Cell 3.')


In [ ]:
# === Verify you are on the clean Unsloth AMD kernel ===
import os, sys
VENV = os.path.expanduser('~/unsloth_gemma4_env')
print('sys.executable:', sys.executable)
print('sys.prefix:', sys.prefix)
print('VIRTUAL_ENV:', os.environ.get('VIRTUAL_ENV'))
on_expected_kernel = (
    sys.prefix == VENV
    or sys.executable.startswith(VENV + os.sep)
    or os.environ.get('VIRTUAL_ENV') == VENV
)
if not on_expected_kernel:
    raise RuntimeError(
        'Wrong kernel. Change kernel to Python (unsloth-gemma4-amd), then restart and re-run this cell.'
    )

import unsloth, unsloth_zoo, huggingface_hub, regex, transformers
from transformers import AutoConfig

print(f'unsloth: {getattr(unsloth, "__version__", "OK")} ({unsloth.__file__})')
print(f'unsloth_zoo: {getattr(unsloth_zoo, "__version__", "OK")} ({unsloth_zoo.__file__})')
print(f'huggingface_hub: {huggingface_hub.__version__} ({huggingface_hub.__file__})')
print(f'regex: {regex.__version__} ({regex.__file__})')
print(f'transformers: {transformers.__version__} ({transformers.__file__})')
c = AutoConfig.from_pretrained("unsloth/gemma-4-31B-it")
print(f'AutoConfig OK: model_type={c.model_type}')

import torch; torch._dynamo.config.recompile_limit = 64
print('Next: run the HF token cell, then the Load Model cell.')


In [ ]:
import os
from pathlib import Path
import huggingface_hub

# ==========================================================
# ⚠️ PASTE YOUR HUGGING FACE TOKEN BETWEEN THE QUOTES BELOW:
# ==========================================================
HF_TOKEN = ""

HF_REPO_ID = "mtita/gemma4-31b-tool-calling-lora"

# Fallback to environment variable if the string above is empty
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN":
    print("⚠️  No HF token set. Hub pushing disabled, and downloading gated models may fail!")
    print("   Please paste your token in the HF_TOKEN variable above.")
    HF_TOKEN = ""
    PUSH_TO_HUB = False
else:
    print("✅ Hugging Face Token recognized!")
    huggingface_hub.login(token=HF_TOKEN)
    PUSH_TO_HUB = True

# === Training Output ===
TRAINING_OUTPUT_DIR = os.environ.get(
    "TRAINING_OUTPUT_DIR",
    str(Path.home() / "gemma4_runs" / "gemma4_toolcalling_sft")
)
Path(TRAINING_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("HF repo:", HF_REPO_ID)
print("Push enabled:", PUSH_TO_HUB)
print("Training outputs:", TRAINING_OUTPUT_DIR)


### Load Model


In [ ]:
from unsloth import FastModel
import torch, os


model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-31B-it",
    dtype = torch.bfloat16, # None for auto detection
    max_seq_length = 4096, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    full_finetuning = False, # LoRA fine-tuning (much less VRAM)
    token = HF_TOKEN or None,
)


### Add LoRA Adapters


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Text-only
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_gradient_checkpointing = True,
)


### Data Prep

We use the `gemma-4` chat template. Gemma 4 renders conversations like:
```
<bos><|turn>user
Hello<turn|>
<|turn>model
Hey there!<turn|>
```


In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)


In [ ]:
from datasets import load_dataset
dataset = load_dataset("mtita/gemma4-tool-calling-training",
                        data_files="combined_training.jsonl",
                        split="train")
print(f"Loaded {len(dataset)} training examples")
print(dataset[0]["messages"][:2])


Map roles for Gemma 4 (system → user prefix, tool → user prefix) and apply chat template.


In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        formatted = []
        for msg in messages:
            role = msg["role"]
            content = msg.get("content", "") or ""
            if role == "system":
                sys_msg = {"role": "user", "content": f"[System Instructions]\n{content}"}
                ack_msg = {"role": "assistant", "content": "Understood."}
                # Merge with previous user if consecutive
                if formatted and formatted[-1]["role"] == "user":
                    formatted[-1]["content"] += f"\n\n{sys_msg['content']}"
                    formatted.append(ack_msg)
                else:
                    formatted.append(sys_msg)
                    formatted.append(ack_msg)
            elif role == "user":
                if formatted and formatted[-1]["role"] == "user":
                    formatted[-1]["content"] += f"\n\n{content}"
                else:
                    formatted.append({"role": "user", "content": content})
            elif role == "assistant":
                if formatted and formatted[-1]["role"] == "assistant":
                    formatted[-1]["content"] += f"\n\n{content}"
                else:
                    formatted.append({"role": "assistant", "content": content})
            elif role == "tool":
                tool_name = msg.get("name", "tool")
                tool_text = f"[Tool Result: {tool_name}]\n{content}"
                if formatted and formatted[-1]["role"] == "user":
                    formatted[-1]["content"] += f"\n\n{tool_text}"
                else:
                    formatted.append({"role": "user", "content": tool_text})
        # Ensure starts with user
        if not formatted or formatted[0]["role"] != "user":
            texts.append("")
            continue
        # Ensure ends with assistant (we want to train on the final response)
        if formatted[-1]["role"] != "assistant":
            texts.append("")
            continue
        try:
            text = tokenizer.apply_chat_template(
                formatted, tokenize=False, add_generation_prompt=False
            ).removeprefix('<bos>')
            texts.append(text)
        except Exception:
            texts.append("")
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
dataset = dataset.filter(lambda x: len(x["text"]) > 50)
print(f"Formatted: {len(dataset)} examples")
print(dataset[0]["text"][:500])

### Train the model


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 4096,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 1e-4,
        logging_steps = 1,
        optim = "adamw_torch",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        output_dir = TRAINING_OUTPUT_DIR,
        save_strategy = "no",
        # save_steps = 50,
        # save_total_limit = 5,
        hub_model_id = HF_REPO_ID,
        push_to_hub = PUSH_TO_HUB,
        hub_strategy = "checkpoint",
    ),
)


Only train on assistant outputs, ignore user inputs:


In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)


In [ ]:
# Verify masking - should see full conversation
tokenizer.decode(trainer.train_dataset[0]["input_ids"])


In [ ]:
# Verify masking - should see ONLY assistant responses (rest is spaces)
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ")


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")


### Train!


In [ ]:
import gc
import os
import json
import shutil
import torch
from huggingface_hub import snapshot_download, HfApi
from transformers.trainer_utils import get_last_checkpoint

def is_checkpoint_valid(ckpt_dir):
    """A checkpoint is only valid if trainer_state.json exists and is readable."""
    if not ckpt_dir or not os.path.isdir(ckpt_dir):
        return False
    state_file = os.path.join(ckpt_dir, 'trainer_state.json')
    if not os.path.isfile(state_file):
        return False
    try:
        with open(state_file) as f:
            json.load(f)
        return True
    except Exception:
        return False

def get_step_from_checkpoint(ckpt_dir):
    if not ckpt_dir: return 0
    state_file = os.path.join(ckpt_dir, 'trainer_state.json')
    if os.path.isfile(state_file):
        try:
            with open(state_file) as f:
                return json.load(f).get('global_step', 0)
        except Exception:
            pass
    # If trainer_state.json is missing/corrupt, this checkpoint is NOT trustworthy.
    # Return 0 so the loop does NOT skip chunks based on a broken checkpoint.
    return 0

def find_best_valid_checkpoint(output_dir):
    """Scan all checkpoint-* folders, remove corrupt ones, return the best valid one."""
    if not os.path.isdir(output_dir):
        return None
    candidates = []
    for name in sorted(os.listdir(output_dir)):
        if not name.startswith('checkpoint-'):
            continue
        full_path = os.path.join(output_dir, name)
        if is_checkpoint_valid(full_path):
            candidates.append(full_path)
        else:
            print(f'\u26a0\ufe0f Removing corrupt/incomplete checkpoint: {name}')
            shutil.rmtree(full_path, ignore_errors=True)
    if not candidates:
        return None
    # Return the one with the highest step number
    return max(candidates, key=lambda p: int(p.rsplit('-', 1)[-1]))

# 0. Sync missing checkpoints from the Hub
if PUSH_TO_HUB:
    print(f'\n\U0001f4e5 Pulling rescued checkpoints from Hugging Face: {HF_REPO_ID}...')
    try:
        snapshot_download(
            repo_id=HF_REPO_ID,
            local_dir=TRAINING_OUTPUT_DIR,
            token=HF_TOKEN,
            allow_patterns=['*checkpoint*']
        )
        print('\u2705 Cloud checkpoints restored to the fresh VM!')
    except Exception as e:
        print('No prior checkpoints found on Hub, starting fresh.')

trainer.args.save_strategy = 'steps'
trainer.args.save_steps = 800
trainer.args.save_total_limit = 2
trainer.args.push_to_hub = False

CHUNK_SIZE = 800
TOTAL_CHUNKS = 22

for i in range(TOTAL_CHUNKS):
    # Always validate - removes corrupt checkpoints automatically
    LAST_CHECKPOINT = find_best_valid_checkpoint(TRAINING_OUTPUT_DIR)

    current_max_steps = (i + 1) * CHUNK_SIZE
    trainer.args.max_steps = current_max_steps

    completed_steps = get_step_from_checkpoint(LAST_CHECKPOINT)
    if LAST_CHECKPOINT and completed_steps >= current_max_steps:
        print(f'\u23ed\ufe0f Skipping chunk {i+1} (Already completed in a previous run)')
        continue

    print(f'\n\U0001f680 Running Chunk {i+1}: Training up to step {current_max_steps}...')
    if LAST_CHECKPOINT:
        print(f'   Resuming from: {os.path.basename(LAST_CHECKPOINT)} (step {completed_steps})')
    else:
        print(f'   No valid checkpoint found, starting from scratch.')

    try:
        trainer_stats = trainer.train(resume_from_checkpoint=LAST_CHECKPOINT)
    except Exception as e:
        print(f'Training interrupted or hit an error: {e}')
        break

    # 100% Dedicated Synchronous Upload
    if PUSH_TO_HUB:
        print(f'\U0001f4e6 Chunk {i+1} finished! Force-uploading local checkpoints to the Cloud...')
        api = HfApi(token=HF_TOKEN)
        api.upload_folder(
            folder_path=TRAINING_OUTPUT_DIR,
            repo_id=HF_REPO_ID,
            repo_type='model',
            allow_patterns=['checkpoint-*/*'],
            commit_message=f'Automated sync reaching step {current_max_steps}'
        )
        print('\u2601\ufe0f Cloud upload totally complete and secured!')

    gc.collect()
    torch.cuda.empty_cache()
    print(f'\U0001f9f9 Chunk {i+1} Memory flushed. Moving to next chunk.')


In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")


### Test Inference


In [ ]:
# Quick inference test
messages = [{"role": "user",
"content": [{"type": "text", "text": "You have these tools:\n"
"- read(path): Read a file\n"
"- edit(path, old_text, new_text): Replace text in a file\n"
"- bash(command): Run a shell command\n\n"
"Fix the typo in src/utils.py where 'retrun' should be 'return'."}]}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 512,
                    temperature = 0.7, top_p = 0.9)

### Save LoRA adapters

**Set your HF token below!**


In [ ]:
model.save_pretrained("gemma_4_lora")
tokenizer.save_pretrained("gemma_4_lora")

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print("Pushed LoRA adapter to", HF_REPO_ID)
else:
    print("Skipping push_to_hub")


### Export to GGUF

Supported: q4_k_m (recommended), q5_k_m, q8_0, f16, bf16 and more.


In [ ]:
model.save_pretrained_gguf(
    "gemma_4_finetune",
    tokenizer,
    quantization_method = "q4_k_m",
)


In [ ]:
# Push GGUF to HuggingFace
if PUSH_TO_HUB:
    model.push_to_hub_gguf(
        "mtita/gemma4-31b-tool-calling-q4_k_m-GGUF",
        tokenizer,
        quantization_method = "q4_k_m",
        token = HF_TOKEN,
    )
    print("Pushed GGUF to Hub")
else:
    print("Saved GGUF locally. Set HF_TOKEN to push.")